# 🧹 Phase 1: Data Cleaning & Pre-processing

Based on the initial analysis, this notebook implements a **conservative cleaning strategy**.
We aim to fix structural errors (missing IDs) and optimize data types while preserving as much information as possible (e.g., keeping anime with partial data).

**Key Actions:**
1.  **Impute** missing values where logic allows (e.g., `favorites=0`, `is_rewatching=False`).
2.  **Drop** only structurally corrupted rows (missing Primary/Foreign Keys).
3.  **Extract** useful features (e.g., `year` from `start_date`).
4.  **Optimize** data types (float64 -> float32/16, objects -> categories) to reduce memory usage.
5.  **Save** to optimized Parquet format for the next steps.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
import gc
from pathlib import Path
import pprint

# 🧪 Standard Import (requires 'solution' package installed)
from solution.src.common import PROCESSED_DATA_DIR
from solution.src.data import load_raw, get_raw_path
from solution.src.utils import save_refined, print_fancy_progress

# 📦 Import atomic cleaning functions and utils
from nb_01_cleaning import (
    REMAINING_FILES,
    optimize_dtypes,
    get_memory_usage_mb,
    clean_string_columns,
    standardize_dataset,
    resolve_content_duplicates,
    print_cleaning_report
)

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print(f"Saving to:    {PROCESSED_DATA_DIR}")

# Global storage for ID remapping (to be applied to relational tables)
GLOBAL_ID_MAP = {}


## 1. Characters (`characters.csv`)
- Drop rows with missing `character_mal_id`.
- Fill missing `favorites` with 0.
- Drop `about` column (mostly empty/text heavy text not needed for general stats).

In [ ]:
# 1. Load
df_chars = load_raw('characters')

# 2. Clean (Conservative)
initial_rows = len(df_chars)

# Remove exact duplicates only
df_chars, std_metrics = standardize_dataset(df_chars)

# Resolve Content Duplicates (Merge IDs)
df_chars, dup_metrics = resolve_content_duplicates(df_chars, primary_key='character_id')

# Store ID map for later use in relational tables
if dup_metrics['id_map']:
    GLOBAL_ID_MAP['character_id'] = dup_metrics['id_map']

# Drop empty rows but NOT just missing IDs
df_chars.dropna(how='all', inplace=True)

df_chars = df_chars.drop(columns=['about'])

rows_dropped_total = initial_rows - len(df_chars)

# 3. Report
report = {
    "Initial Rows": initial_rows,
    "Standardization": std_metrics,
    "Content Resolution": dup_metrics,
    "Final Rows": len(df_chars),
    "Total Dropped": rows_dropped_total
}

print_cleaning_report("Characters", report)

# 4. Optimize & Save
df_chars = optimize_dtypes(df_chars)
save_refined(df_chars, PROCESSED_DATA_DIR / "characters.parquet", format='parquet')

# 5. Capture Valid Character IDs
valid_character_ids = set(df_chars['character_id'].dropna())
print(f"Captured {len(valid_character_ids)} valid Character IDs.")


## 2. Anime Details (`details.csv`)
- **Year**: Derive from `start_date` where missing.
- **Dates**: Convert to datetime objects.
- **Missing Stats**: Keep `score`, `rank` as NaN (preserve "unranked" status).
- **Columns**: Keep `demographics` and others even if they look empty (might have sparse data).

In [ ]:
# 1. Load
df_details = load_raw('details')

# 2. Clean (Conservative)
initial_rows = len(df_details)

# Standardize
df_details, std_metrics = standardize_dataset(df_details)

# Resolve Content Duplicates
# Check if resolve_content_duplicates is imported (it should be from previous patch)
# If not, the notebook run might fail, but we assume previous patch worked.
df_details, dup_metrics = resolve_content_duplicates(df_details, primary_key='anime_id')
if dup_metrics['id_map']:
    GLOBAL_ID_MAP['anime_id'] = dup_metrics['id_map']

df_details.dropna(how='all', inplace=True)

# Features
for col in ['start_date', 'end_date']:
    df_details[col] = pd.to_datetime(df_details[col], errors='coerce')

# --- Imputation: meaningful calculations ---
# Calculate Year from start_date if missing
missing_year_before = df_details['year'].isna().sum() if 'year' in df_details.columns else 0
df_details['year'] = df_details['year'].fillna(df_details['start_date'].dt.year)
missing_year_after = df_details['year'].isna().sum() if 'year' in df_details.columns else 0
year_imputed_count = missing_year_before - missing_year_after
# -------------------------------------------

cat_cols = ['type', 'status', 'source', 'rating', 'season']
for col in cat_cols:
    if col in df_details.columns:
        df_details[col] = df_details[col].astype('category')

report = {
    "Initial Rows": initial_rows,
    "Standardization": std_metrics,
    "Content Resolution": dup_metrics,
    "Final Rows": len(df_details),
    "Year Imputed": year_imputed_count
}

print_cleaning_report("Details", report)

# 3. Optimize & Save
df_details = optimize_dtypes(df_details)
save_refined(df_details, PROCESSED_DATA_DIR / "details.parquet", format='parquet')

# 4. Capture Valid Anime IDs
valid_anime_ids = set(df_details['anime_id'].dropna())
print(f"Captured {len(valid_anime_ids)} valid Anime IDs.")


## 3. Anime Stats (`stats.csv`)
- **Votes**: Fill NaN score votes with 0.
- **Percentages**: Fill NaN percentages with 0.
- **Conserve Rows**: Do not drop rows just because they have no votes.

In [ ]:
# 1. Load
df_stats = load_raw('stats')

# 2. Clean
initial_rows = len(df_stats)

df_stats, std_metrics = standardize_dataset(df_stats)
df_stats, dup_metrics = resolve_content_duplicates(df_stats, primary_key='anime_id')
# Merge maps if any found here (unlikely if unique per anime)
if dup_metrics['id_map']:
    if 'anime_id' not in GLOBAL_ID_MAP:
        GLOBAL_ID_MAP['anime_id'] = {}
    GLOBAL_ID_MAP['anime_id'].update(dup_metrics['id_map'])

df_stats.dropna(how='all', inplace=True)

report = {
    "Initial Rows": initial_rows,
    "Standardization": std_metrics,
    "Content Resolution": dup_metrics,
    "Final Rows": len(df_stats)
}
print_cleaning_report("Stats", report)

# 3. Optimize & Save
df_stats = optimize_dtypes(df_stats)
save_refined(df_stats, PROCESSED_DATA_DIR / "stats.parquet", format='parquet')


## 4. User Ratings (`ratings.csv`)
- **Huge File**: Process carefully.
- **Username**: Drop NaN (orphan votes).
- **Rewatching**: Assume `False` (0) if missing.

In [ ]:
chunk_size = 5_000_000
chunks = []
csv_path = get_raw_path('ratings')

print(f"Processing {csv_path} in chunks with Conservative Logic...")

id_map_anime = GLOBAL_ID_MAP.get('anime_id', {})
total_remapped = 0

total_metrics = {
    'rows_read': 0,
    'dropped_orphans': 0,
    'rewatch_kept_nan': 0
}

with pd.read_csv(csv_path, chunksize=chunk_size) as reader:
    for i, chunk in enumerate(reader, 1):
        print_fancy_progress(i, 0, f'Chunk {i}: ') # reader len is unknown with chunksize sometimes?

        metrics = {'rows_read': len(chunk)}

        # 1. Remap FKs
        if id_map_anime:
            # Only apply if overlaps
            # Optimized replace: using map is faster than replace for large dicts
            # But map turns non-matches to NaN, so require combine_first or careful app
            # For safe partial replacement:
            mask = chunk['anime_id'].isin(id_map_anime.keys())
            if mask.any():
                chunk.loc[mask, 'anime_id'] = chunk.loc[mask, 'anime_id'].map(id_map_anime)
                total_remapped += mask.sum()

        # 2. Filter Orphans (Strict Referential Integrity)
        chunk = chunk[chunk['anime_id'].isin(valid_anime_ids)]
        metrics['dropped_orphans'] = metrics['rows_read'] - len(chunk)

        # 3. No Imputation
        # chunk['is_rewatching'] = ... 

        # 4. Optimize Types
        dtype_map = {
            'anime_id': 'Int32', # Int32 allows NaNs, normal int32 doesn't
            'score': 'Int8',
            'num_watched_episodes': 'Int32', 
            'status': 'category'
        }

        total_metrics['rows_read'] += metrics['rows_read']
        total_metrics['dropped_orphans'] += metrics['dropped_orphans']

        chunks.append(chunk)

print("\nConcatenating processed chunks...")
df_ratings = pd.concat(chunks, ignore_index=True)

final_report = {
    "Total Rows Read": total_metrics['rows_read'],
    "Orphans Dropped": total_metrics['dropped_orphans'],
    "FKs Remapped": total_remapped,
    "Final Rows": len(df_ratings)
}

print_cleaning_report("Ratings", final_report)

df_ratings = optimize_dtypes(df_ratings)
save_refined(df_ratings, PROCESSED_DATA_DIR / "ratings.parquet", format='parquet')

del df_ratings, chunks
_ = gc.collect()


## 5. Other Files
Processing the remaining files with standard optimizations:
- `person_details.csv`
- `profiles.csv`
- `favs.csv`
- `recommendations.csv`
- Relational tables (`*_works.csv`, `*_nicknames.csv`)

In [ ]:
results = []
print(f"Processing {len(REMAINING_FILES)} other files...")

valid_ids_map = {}
valid_ids_map['anime_id'] = valid_anime_ids
# We use 'character_id' because files are renamed to standardize IDs
valid_ids_map['character_id'] = valid_character_ids

for i, dataset_name in enumerate(REMAINING_FILES, 1):
    print(f"Processing {dataset_name}...")
    # 1. Read
    df = load_raw(dataset_name)

    # Stats Capture
    initial_rows = len(df)
    initial_mem = get_memory_usage_mb(df)

    # 2. Clean (Generic Logic Inlined)
    actions = []

    df, metrics = standardize_dataset(df)

    if metrics['duplicates_dropped'] > 0:
        actions.append(f"Deduplicated ({metrics['duplicates_dropped']} rows)")

    df.dropna(how='all', inplace=True)

    if dataset_name == "person_details":
        df['birthday'] = pd.to_datetime(df['birthday'], errors='coerce')
        actions.append("Date Parsing (birthday)")

    if dataset_name == "profiles":
        for date_col in ['joined']:
            df['joined'] = pd.to_datetime(df['joined'], errors='coerce')
            actions.append(f"Date Parsing ({'joined'})")
    # -----------------------------

    # 3. Optimize
    df = optimize_dtypes(df, verbose=False)

    # Capture Final Stats
    final_rows = len(df)
    final_mem = get_memory_usage_mb(df)

    # 4. Save
    # We use file_path.name to get the filename like 'person_details.csv' then replace extension
    out_name = file_path.name.replace('.csv', '.parquet')
    save_refined(df, PROCESSED_DATA_DIR / out_name, format='parquet')

    results.append({
        'file': dataset_name, 
        'rows_src': initial_rows, 
        'rows_dst': final_rows,
        'mem_src': initial_mem,
        'mem_dst': final_mem,
        'actions': actions
    })

# Print Summary
print(f"\n--- Other Files Cleaning Report ---")
for res in results:
    mem_red = (res['mem_src'] - res['mem_dst']) / res['mem_src'] * 100 if res['mem_src'] > 0 else 0
    print(f"[{res['file']}]")
    print(f"  Rows:   {res['rows_src']} -> {res['rows_dst']}")
    print(f"  Memory: {res['mem_src']:.2f} MB -> {res['mem_dst']:.2f} MB ({mem_red:.1f}% reduction)")
    if res['actions']:
        print(f"  Actions: {', '.join(res['actions'])}")
    print("-" * 30)
print(f"Total files processed: {len(results)}")
